In [10]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd

In [11]:
import sys
from pathlib import Path

DB_NAME = "olist_db"

project_root = Path(sys.path[0])

def make_url(database):
    return URL.create(
        "mssql+pyodbc",
        host="localhost",
        database=database,
        query={
            "driver": "ODBC Driver 18 for SQL Server",
            "TrustServerCertificate": "yes",
            "Trusted_Connection": "yes",
        },
    )

In [15]:
engine = create_engine(make_url(DB_NAME))

#1-Loading the geolocation table

df_geolocation = pd.read_csv(
    f"{project_root}/cleaned_data/geolocation.csv",
    usecols=["zip_code", "city", "state"],
    dtype=str
)
df_geolocation_mapped = df_geolocation.rename(
    columns={
        "zip_code":"zip_code_prefix",
    }
)
df_geolocation_mapped["zip_code_prefix"] = df_geolocation_mapped["zip_code_prefix"].str.strip().str.zfill(5)
df_geolocation_mapped.to_sql("dim_geolocations",engine,if_exists="append",index=False)

399

In [18]:
import os

# --- Loading the dim_dates table ---
raw_dir = f"{project_root}/raw_dataset"
files = [f for f in os.listdir(raw_dir) if f.endswith('.csv')]
all_dates = []

for file in files:
    df = pd.read_csv(os.path.join(raw_dir, file))
    date_cols = [col for col in df.columns if 'date' in col.lower() or 'timestamp' in col.lower()]
    for col in date_cols:
        dates = pd.to_datetime(df[col], errors='coerce').dropna().dt.date.unique()
        all_dates.extend(dates)

unique_dates = sorted(list(set(all_dates)))
df_dates_raw = pd.DataFrame({'full_date': unique_dates})
df_dates_raw['full_date'] = pd.to_datetime(df_dates_raw['full_date'])

# Create date dimensions based on DimDate model
df_dates = pd.DataFrame({
    'date_key': df_dates_raw['full_date'].dt.strftime('%Y%m%d').astype(int),
    'full_date': df_dates_raw['full_date'].dt.date,
    'year': df_dates_raw['full_date'].dt.year,
    'quarter': df_dates_raw['full_date'].dt.quarter,
    'month': df_dates_raw['full_date'].dt.month,
    'day': df_dates_raw['full_date'].dt.day,
    'day_of_week': df_dates_raw['full_date'].dt.dayofweek
})

# Handle SQL Server IDENTITY_INSERT for the date_key primary key
with engine.begin() as conn:
    conn.execute(text("SET IDENTITY_INSERT dim_dates ON"))
    df_dates.to_sql("dim_dates", conn, if_exists="append", index=False)
    conn.execute(text("SET IDENTITY_INSERT dim_dates OFF"))

print(f"Loaded {len(df_dates)} unique dates into dim_dates with identity insert handled.")


C:\Users\SUANON O.K Fadil\AppData\Local\Temp\ipykernel_12976\423497997.py:27: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  'day_of_week': df_dates_raw['full_date'].dt.dayofweek


Loaded 755 unique dates into dim_dates with identity insert handled.
